**Load features, define X/y, and do the train/test split**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("beard_model_ready_features.csv")

feature_cols = [
    "HeavyAtomCount", "LargestConjugatedSystemSize", "TPSA",
    "NumHDonors", "NumRotatableBonds", "FractionCSP3", "MolLogP_clipped",
    "NumDonorGroups", "NumAcceptorGroups", "HasPushPull"
]

X = df[feature_cols]
y = df["lambda_max_exp_nm"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train.shape[0]} compounds")
print(f"Test set: {X_test.shape[0]} compounds")
print(f"Train y range: {y_train.min():.1f} - {y_train.max():.1f}, mean {y_train.mean():.1f}")
print(f"Test y range: {y_test.min():.1f} - {y_test.max():.1f}, mean {y_test.mean():.1f}")

Train set: 5502 compounds
Test set: 1376 compounds
Train y range: 200.0 - 900.0, mean 410.0
Test y range: 200.0 - 900.0, mean 406.4


**Scale features correctly — fit only on training data**

In [2]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # fit AND transform on train
X_test_scaled = scaler.transform(X_test)          # transform only, using train's fitted parameters

print("Scaler fitted on training data only.")
print(f"Train mean (should be ~0): {X_train_scaled.mean(axis=0).round(3)}")
print(f"Train std (should be ~1): {X_train_scaled.std(axis=0).round(3)}")
print(f"Test mean (will NOT be exactly 0 - expected): {X_test_scaled.mean(axis=0).round(3)}")

Scaler fitted on training data only.
Train mean (should be ~0): [-0.  0.  0.  0. -0.  0. -0.  0. -0. -0.]
Train std (should be ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Test mean (will NOT be exactly 0 - expected): [ 0.013 -0.002 -0.044 -0.02  -0.01  -0.002  0.034 -0.028 -0.055 -0.036]


**Fit both OLS and Ridge, and actually look at coefficient stability**

In [3]:
from sklearn.linear_model import LinearRegression, Ridge
import numpy as np

ols = LinearRegression()
ols.fit(X_train_scaled, y_train)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

coef_comparison = pd.DataFrame({
    "feature": feature_cols,
    "OLS_coef": ols.coef_,
    "Ridge_coef": ridge.coef_
})
print(coef_comparison.to_string(index=False))

                    feature  OLS_coef  Ridge_coef
             HeavyAtomCount -0.314648   -0.301437
LargestConjugatedSystemSize 33.360041   33.325578
                       TPSA -0.883327   -0.873024
                 NumHDonors -4.502656   -4.501283
          NumRotatableBonds -4.969790   -4.972729
               FractionCSP3 12.355132   12.345287
            MolLogP_clipped -8.464371   -8.445283
             NumDonorGroups 13.129265   13.122881
          NumAcceptorGroups 13.444155   13.435667
                HasPushPull -1.351904   -1.346972


**Actually evaluate on the held-out test set**

In [4]:
from sklearn.metrics import mean_squared_error, r2_score

for name, model in [("OLS", ols), ("Ridge", ridge)]:
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{name}: RMSE = {rmse:.2f} nm, R² = {r2:.3f}")

OLS: RMSE = 121.58 nm, R² = 0.052
Ridge: RMSE = 121.58 nm, R² = 0.052


**null baseline**

In [5]:
# Null baseline: predict the training mean for every test compound, no features used at all
naive_pred = np.full_like(y_test, y_train.mean(), dtype=float)
naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
naive_r2 = r2_score(y_test, naive_pred)

print(f"Naive (mean-prediction) baseline: RMSE = {naive_rmse:.2f} nm, R² = {naive_r2:.3f}")
print(f"Ridge model:                       RMSE = 121.58 nm, R² = 0.052")

Naive (mean-prediction) baseline: RMSE = 124.89 nm, R² = -0.001
Ridge model:                       RMSE = 121.58 nm, R² = 0.052


In [6]:
from sklearn.linear_model import RidgeCV

alphas_to_try = [0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0]

ridge_cv = RidgeCV(alphas=alphas_to_try, cv=5)
ridge_cv.fit(X_train_scaled, y_train)

print(f"Best alpha selected by cross-validation: {ridge_cv.alpha_}")

y_pred_cv = ridge_cv.predict(X_test_scaled)
rmse_cv = np.sqrt(mean_squared_error(y_test, y_pred_cv))
r2_cv = r2_score(y_test, y_pred_cv)
print(f"Tuned Ridge: RMSE = {rmse_cv:.2f} nm, R² = {r2_cv:.3f}")

Best alpha selected by cross-validation: 100.0
Tuned Ridge: RMSE = 121.67 nm, R² = 0.050
